Environment setup

We used Google Colab to run the lab. Spark was deployed locally by installing OpenJDK 17, downloading and extracting Spark 4.0.1 (Hadoop 3 build), and setting JAVA_HOME (auto-detected) and SPARK_HOME.

In [1]:
# install openjdk (use 17 in Colab; Java8 on Colab )
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null

# check java version
!java -version

# download spark4.0.1
!wget -q https://archive.apache.org/dist/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz

# unzip it
!tar xf spark-4.0.1-bin-hadoop3.tgz

import os
import sys
import time
import subprocess

# set JAVA_HOME correctly (auto-detect) ---
java_path = subprocess.check_output(["bash", "-lc", "readlink -f $(which java)"]).decode().strip()
os.environ["JAVA_HOME"] = java_path.replace("/bin/java", "")


os.environ["SPARK_HOME"] = "/content/spark-4.0.1-bin-hadoop3"

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("SPARK_HOME =", os.environ["SPARK_HOME"])


# check java version
!java -version

# match pyspark version to spark ---
!pip -q install pyspark==4.0.1
import pyspark
print("pyspark =", pyspark.__version__)

# install findspark
!pip -q install findspark
import findspark

# Initialize findspark to locate Spark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.sql.functions import *

# Create a SparkConf object to set properties
conf = SparkConf()
conf.set("spark.driver.memory", "4g")

spark = SparkSession.builder.master("local[*]").appName("TextAnalytics").config(conf=conf).getOrCreate()
sc = spark.sparkContext

spark


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)
JAVA_HOME = /usr/lib/jvm/java-17-openjdk-amd64
SPARK_HOME = /content/spark-4.0.1-bin-hadoop3
openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)
pyspark = 4.0.1


So far, our enviroment set up is successful, then we can start our Lab queston.

In [2]:
# Exercise 0 - Download and load Shakespeare corpus

#It hasn't been downloaded yet; it's just being prepared for URL.
from pyspark import SparkFiles
url_romeo_juliet = "https://www.gutenberg.org/cache/epub/1112/pg1112.txt"
url_hamlet = "https://www.gutenberg.org/files/1524/1524-0.txt"
url_richard_ii = "https://www.gutenberg.org/cache/epub/1776/pg1776.txt"

# Add remote files to Spark
sc.addFile(url_romeo_juliet)
sc.addFile(url_hamlet)
sc.addFile(url_richard_ii)

# Load each text as an RDD (one line per element)
romeo_juliet_rdd = sc.textFile(SparkFiles.get("pg1112.txt"))
hamlet_rdd = sc.textFile(SparkFiles.get("1524-0.txt"))
richard_ii_rdd = sc.textFile(SparkFiles.get("pg1776.txt"))

# Quick sanity check: display a few lines (action)
romeo_juliet_rdd.take(5)
hamlet_rdd.take(5)
richard_ii_rdd.take(5)


['The Project Gutenberg eBook of King Richard II',
 '    ',
 'This ebook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms']

The process in this step is as follows: Download the texts of 3 Shakespeare books to the Spark environment,read them as an RDD (Resilient Distributed Dataset),preparing for subsequent map/flatMap/reduce operations.

In [5]:
# Exercise 1 (4 points):  Count Words
# 1) Cleaning function (per line)
#    - use translate() to remove punctuation (as required)
import string, re
PUNCT_TABLE = str.maketrans('', '', string.punctuation)

def clean_line(line: str) -> str:
    return line.translate(PUNCT_TABLE).lower()

# 2) Token filter (letters only)
WORD_RE = re.compile(r"^[a-z]+$")

def filter_word(w: str) -> bool:
    return WORD_RE.match(w) is not None


# 3) WordCount pipeline (RDD -> PairRDD(word, count))
#    Uses: flatMap/split, filter/re.match, map, reduceByKey
def wordcount_rdd(lines_rdd):
    return (lines_rdd
            .map(clean_line)                      # clean each line once
            .flatMap(lambda line: line.split())   # tokenize
            .filter(filter_word)                # keep only [a-z]+
            .map(lambda w: (w, 1))
            .reduceByKey(lambda a, b: a + b)
           )

# Apply to the 3 corpora
romeo_wc   = wordcount_rdd(romeo_juliet_rdd)
hamlet_wc  = wordcount_rdd(hamlet_rdd)
richard_wc = wordcount_rdd(richard_ii_rdd)

# check (action)
romeo_wc.take(20)
hamlet_wc.take(20)
richard_wc.take(20)


[('of', 749),
 ('king', 259),
 ('richard', 143),
 ('this', 249),
 ('for', 261),
 ('use', 25),
 ('anyone', 6),
 ('united', 15),
 ('states', 20),
 ('and', 862),
 ('most', 23),
 ('other', 41),
 ('world', 33),
 ('at', 82),
 ('no', 109),
 ('cost', 5),
 ('with', 304),
 ('almost', 2),
 ('restrictions', 2),
 ('whatsoever', 2)]

In [10]:
# Exercise 2 (6 points): Frequent Terms & Stop Words
# Goal: merge stopwords of 3 documents, sum frequencies, sort desc, save ONE csv

import nltk
from nltk.corpus import stopwords

# 1) get English stopwords from nltk
nltk.download("stopwords", quiet=True)
stopwords_set = set(stopwords.words("english"))

# 2) filter(): keep only stopwords in each (word, count) RDD
s1_sw = romeo_wc.filter(lambda x: x[0] in stopwords_set)
s2_sw = hamlet_wc.filter(lambda x: x[0] in stopwords_set)
s3_sw = richard_wc.filter(lambda x: x[0] in stopwords_set)

# 3) union() & reduceByKey(): merge stopwords and sum frequencies
all_sw = s1_sw.union(s2_sw).union(s3_sw)                 # (word, count)
all_sw_sum = all_sw.reduceByKey(lambda a, b: a + b)      # (word, total_frequency)

# 4) map(): swap to (freq, word) so we can sortByKey by frequency
sw_sorted = (all_sw_sum
             .map(lambda x: (x[1], x[0]))
             .sortByKey(ascending=False)                 # desc freq
             .map(lambda y: (y[0], y[1]))             # keep (Frequency, Word)
            )

# 5) spark.createDataFrame(): create dataframe and show table
df = spark.createDataFrame(sw_sorted, ["Frequency", "Word"])
df.show(10, truncate=False)

# 6) write.csv(): store into a single csv file
output_path = "/content/stopwords_frequencies"
(df.coalesce(1)
   .write
   .mode("overwrite")
   .option("header", "true")
   .csv(output_path))

print(f"Saved CSV folder to: {output_path}")
print("Open the folder and download the single 'part-*.csv' file inside that folder.")


+---------+----+
|Frequency|Word|
+---------+----+
|3024     |the |
|2622     |and |
|2067     |to  |
|1957     |of  |
|1610     |i   |
|1455     |a   |
|1358     |my  |
|1201     |in  |
|1191     |you |
|1069     |that|
+---------+----+
only showing top 10 rows
Saved CSV folder to: /content/stopwords_frequencies
Open the folder and download the single 'part-*.csv' file inside that folder.


In [11]:
# Exercise 3 (5 points): Simple Inverted Index
# Build inverted index after skipping stopwords.
# Output: (ID, Term, Document-list)

import nltk
from nltk.corpus import stopwords

# 1) stopwords
nltk.download("stopwords", quiet=True)
stop_words_set = set(stopwords.words("english"))

# 2) Reuse cleaning/token rules from Exercise 1
def doc_unique_terms(lines_rdd):

    return (lines_rdd
            .map(clean_line)                              # reuse translate+lower
            .flatMap(lambda line: line.split())           # tokenize
            .filter(filter_word)                        # keep [a-z]+
            .filter(lambda w: w not in stop_words_set)    # skip stopwords
            .distinct()                                   # unique terms in that doc
           )

# 3) unique terms per document
terms_rj = doc_unique_terms(romeo_juliet_rdd)
terms_h  = doc_unique_terms(hamlet_rdd)
terms_r2 = doc_unique_terms(richard_ii_rdd)

# 4) map(): label each term with its document filename
doc1 = terms_rj.map(lambda w: (w, "RomeoJuliet.txt"))
doc2 = terms_h.map(lambda w: (w, "Hamlet.txt"))
doc3 = terms_r2.map(lambda w: (w, "RichardII.txt"))

word_doc_pairs = doc1.union(doc2).union(doc3)  # (term, doc)

# 5) Count distinct terms in whole corpus (after stopword removal)
distinct_word_count = word_doc_pairs.keys().distinct().count()
print("Number of distinct (non-stopword) words in corpus =", distinct_word_count)

# 6) Inverted index: recommended implementation (reduceByKey)
#    Build sets to avoid duplicates, then convert to sorted list for display.
inverted_rbk = (word_doc_pairs
                .mapValues(lambda d: {d})                   # (term, {doc})
                .reduceByKey(lambda a, b: a.union(b))       # merge sets
                .mapValues(lambda s: sorted(s))             # list for deterministic output
               )

# (Optional) Alternative with groupByKey (less efficient, but good for "experiment" requirement)
inverted_gbk = (word_doc_pairs
                .groupByKey()
                .mapValues(lambda docs: sorted(set(docs)))
               )

# Choose final
inverted_final = inverted_rbk

# 7) Prepare table: (Term, "doc1, doc2, ...")
inverted_for_table = inverted_final.map(lambda x: (x[0], ", ".join(x[1])))

# 8) zipWithIndex(): add an ID column starting from 1
with_id = (inverted_for_table
           .zipWithIndex()
           .map(lambda x: (x[1] + 1, x[0][0], x[0][1]))     # (ID, Term, Document)
          )

# 9) Show result as DataFrame (teacher-style table)
df_index = spark.createDataFrame(with_id, ["ID", "Term", "Document"])
df_index.show(20, truncate=False)


Number of distinct (non-stopword) words in corpus = 9090
+---+------------+------------------------------------------+
|ID |Term        |Document                                  |
+---+------------+------------------------------------------+
|1  |romeo       |RomeoJuliet.txt                           |
|2  |anyone      |RichardII.txt, RomeoJuliet.txt            |
|3  |united      |RichardII.txt, RomeoJuliet.txt            |
|4  |world       |Hamlet.txt, RichardII.txt, RomeoJuliet.txt|
|5  |almost      |Hamlet.txt, RichardII.txt, RomeoJuliet.txt|
|6  |restrictions|RichardII.txt, RomeoJuliet.txt            |
|7  |whatsoever  |Hamlet.txt, RichardII.txt, RomeoJuliet.txt|
|8  |reuse       |RichardII.txt, RomeoJuliet.txt            |
|9  |country     |Hamlet.txt, RichardII.txt, RomeoJuliet.txt|
|10 |william     |Hamlet.txt, RichardII.txt, RomeoJuliet.txt|
|11 |november    |RomeoJuliet.txt                           |
|12 |updated     |RichardII.txt, RomeoJuliet.txt            |
|13 |start   

In [14]:
# Exercise 4 (3 points): Extended Inverted Index
# Keep per-document frequency and format: "doc.txt #freq, ..."
# Sort the document list by Frequency (desc).

import nltk
from nltk.corpus import stopwords
from operator import add

nltk.download("stopwords", quiet=True)
stop_words_set = set(stopwords.words("english"))

# NOTE:
# We reuse from Exercise 1:
# - clean_line(line): translate punctuation + lowercase
# (So we don't re-implement regex logic and we keep consistent preprocessing.)

def doc_wordcount_txt(lines_rdd, filename):
    """
    Return RDD[(term, (filename, freq_in_that_doc))]
    - clean + tokenize
    - keep alphabetic words only
    - skip stopwords
    - count frequency inside ONE document
    """
    word_count = (lines_rdd
          .map(clean_line)
          .flatMap(lambda line: line.split())
          .filter(filter_word)
          .filter(lambda x: x not in stop_words_set)
          .map(lambda x: (x, 1))
          .reduceByKey(add))                          # (term, freq)

    return word_count.map(lambda kv: (kv[0], (filename, kv[1])))


# 1) build per-doc (term -> (doc, freq))
txt1 = doc_wordcount_txt(romeo_juliet_rdd, "RomeoJuliet.txt")
txt2 = doc_wordcount_txt(hamlet_rdd,       "Hamlet.txt")
txt3 = doc_wordcount_txt(richard_ii_rdd,   "RichardII.txt")

# 2) union all
all_txt = txt1.union(txt2).union(txt3)         # (term, (doc, freq))

# 3) reduceByKey(): collect list of (doc, freq) for each term
by_term = (all_txt
           .mapValues(lambda df: [df])                 # (term, [(doc,freq)])
           .reduceByKey(lambda a, b: a + b))           # (term, [(doc,freq), ...])

# 4) sort doc list by freq desc + format "doc #freq"
def format_doc_list(doc_freq_list):
    doc_freq_sorted = sorted(doc_freq_list, key=lambda x: (-x[1], x[0]))  # desc freq, tie -> doc name
    return ", ".join([f"{doc} #{freq}" for doc, freq in doc_freq_sorted])

extended_index = by_term.mapValues(format_doc_list)    # (term, "doc #f, doc #f,...")

# Optional: keep IDs stable across runs
extended_index_sorted = extended_index.sortByKey(ascending=True)

# 5) zipWithIndex(): add ID column
with_id = (extended_index_sorted
           .zipWithIndex()
           .map(lambda x: (x[1] + 1, x[0][0], x[0][1])))   # (ID, Term, Document #Frequency)

# 6) show as DataFrame table
df_ext = spark.createDataFrame(with_id, ["ID", "Term", "Document #Frequency"])
df_ext.show(20, truncate=False)


+---+-------------+---------------------------------------------------+
|ID |Term         |Document #Frequency                                |
+---+-------------+---------------------------------------------------+
|1  |abate        |Hamlet.txt #1, RomeoJuliet.txt #1                  |
|2  |abatements   |Hamlet.txt #1                                      |
|3  |abbey        |RomeoJuliet.txt #1                                 |
|4  |abbot        |RichardII.txt #7                                   |
|5  |abbreviations|RomeoJuliet.txt #1                                 |
|6  |abels        |RichardII.txt #1                                   |
|7  |abet         |RichardII.txt #1                                   |
|8  |abhorred     |Hamlet.txt #1, RomeoJuliet.txt #1                  |
|9  |abhors       |RomeoJuliet.txt #1                                 |
|10 |abide        |RichardII.txt #2, RomeoJuliet.txt #1               |
|11 |able         |RomeoJuliet.txt #2, Hamlet.txt #1, RichardII.